# Fase 2: Integración de Datos de Stack Overflow Developer Surveys

Este cuaderno realiza la integración de una de las fuentes secundarias más críticas para el proyecto "Pulso Tecnológico": los **Stack Overflow Developer Surveys (2017–2024)**. 

El objetivo principal es consolidar 8 años de encuestas históricas para obtener las tendencias de uso y deseo de lenguajes de programación. Dado que el esquema de la encuesta varía con los años y la cantidad de encuestados fluctúa, se vuelve indispensable:

1. **Homogeneizar Esquemas:** Unificar las columnas que cambiaron de nombre para poder realizar un análisis longitudinal.
2. **Calcular Porcentajes en lugar de Conteos Absolutos:** Como el tamaño de la muestra varía sustancialmente año con año, trabajar con recuentos absolutos distorsionaría la tendencia real. Por ello, es imperativo calcular el porcentaje de encuestados que usan o desean una tecnología respecto al total de encuestados del año en curso.

Todo el procesamiento se lleva a cabo utilizando la API de Polars, apalancándose en la ejecución diferida (`LazyFrames`), vectorización y paralelismo para lograr un desempeño óptimo y bajo uso de memoria.

In [2]:
import polars as pl
from pathlib import Path

# Configuraciones de Polars para mejorar la visualización si se muestran DataFrames
pl.Config.set_fmt_str_lengths(50)
pl.Config.set_tbl_rows(10)

polars.config.Config

## Sección 1 — Ingestión Iterativa con Etiqueta de Año

Los datos crudos históricos se encuentran almacenados por año. En esta sección iteramos sobre los años, creamos un plan de ejecución diferido (`scan_csv`) por cada archivo e inyectamos dinámicamente la columna `year`. Finalmente, realizamos una concatenación diagonal, la cual soporta la combinación de DataFrames con esquemas heterogéneos rellenando las columnas faltantes con nulos.

In [4]:
RAW_DATA_DIR = Path("../data/datos_crudos/survey")

lazy_frames = []

for year in range(2017, 2025):
    file_path = RAW_DATA_DIR / f"survey_{year}.csv"
    if file_path.exists():
        # Utilizamos infer_schema_length=0 para tratar temporalmente todo como String
        # y evitar conflictos de tipos durante la concatenación diagonal de 8 años de esquemas distintos.
        lf = pl.scan_csv(file_path, infer_schema_length=0, ignore_errors=True)
        # Inyectamos la columna year dinámicamente
        lf = lf.with_columns(pl.lit(year).cast(pl.Int32).alias("year"))
        lazy_frames.append(lf)
    else:
        print(f"Advertencia: No se encontró el archivo para el año {year} en la ruta {file_path}")

# Concatenamos de forma diagonal todos los DataFrames perezosos
df_lazy_raw = pl.concat(lazy_frames, how="diagonal")

## Sección 2 — Homogeneización de Esquemas

A lo largo de 8 años, Stack Overflow ha cambiado sutilmente los nombres de las columnas que representan los lenguajes utilizados y deseados. Aquí aplicamos un diccionario de mapeo usando la función `pl.coalesce`, la cual nos permite fusionar múltiples columnas retornando el primer valor no nulo de la lista.

In [5]:
COL_MAP = {
    "HaveWorkedLanguage":      "used_langs",   # 2017
    "LanguageWorkedWith":      "used_langs",   # 2018–2019
    "LanguageHaveWorkedWith":  "used_langs",   # 2020–2024
    "WantWorkLanguage":        "wanted_langs", # 2017
    "LanguageDesireNextYear":  "wanted_langs", # 2018–2019
    "LanguageWantToWorkWith":  "wanted_langs", # 2020–2024
}

# Extraemos todas las columnas actuales del esquema lazy para aplicar coalesce
cols_available = df_lazy_raw.collect_schema().names()

# Incluimos explícitamente las columnas de 2017 en la lista de búsqueda
used_cols = [c for c in ["HaveWorkedLanguage", "LanguageWorkedWith", "LanguageHaveWorkedWith"] if c in cols_available]
wanted_cols = [c for c in ["WantWorkLanguage", "LanguageDesireNextYear", "LanguageWantToWorkWith"] if c in cols_available]

df_homogenized = df_lazy_raw.with_columns(
    used_langs=pl.coalesce(used_cols),
    wanted_langs=pl.coalesce(wanted_cols)
).select(["year", "used_langs", "wanted_langs"])

## Sección 3 — Explosión, Normalización y Agregación

Las respuestas contienen múltiples lenguajes concatenados por punto y coma (`;`). Para su correcta agregación debemos:
1. Dividir las cadenas de texto (`.str.split(";")`).
2. Desanidar la lista generada en múltiples filas (`.explode()`).
3. Normalizar el texto (convertir a minúsculas y quitar espacios en blanco laterales).

Para obtener métricas representativas, el análisis no debe quedarse en recuentos absolutos (dado que los encuestados anuales fluctúan), sino basarse en **porcentajes de adopción y deseo** relativos al tamaño de la muestra de cada año.

In [6]:
# Calculamos el total de encuestados por año antes de la explosión
total_respondents = df_homogenized.group_by("year").agg(pl.len().alias("n_respondents"))

# Pipeline para lenguajes usados
used_df = (
    df_homogenized.select(["year", "used_langs"])
    .drop_nulls("used_langs")
    .with_columns(pl.col("used_langs").str.split(";"))
    .explode("used_langs")
    .with_columns(pl.col("used_langs").str.to_lowercase().str.strip_chars().alias("tag"))
    .group_by(["year", "tag"])
    .agg(pl.len().alias("used_count"))
)

# Pipeline para lenguajes deseados
wanted_df = (
    df_homogenized.select(["year", "wanted_langs"])
    .drop_nulls("wanted_langs")
    .with_columns(pl.col("wanted_langs").str.split(";"))
    .explode("wanted_langs")
    .with_columns(pl.col("wanted_langs").str.to_lowercase().str.strip_chars().alias("tag"))
    .group_by(["year", "tag"])
    .agg(pl.len().alias("wanted_count"))
)

# Unimos ambos resultados y calculamos los porcentajes relativos a cada año
final_lazy_query = (
    used_df.join(wanted_df, on=["year", "tag"], how="full", coalesce=True)
    .fill_null(0)
    .join(total_respondents, on="year", how="left")
    .with_columns(
        used_pct=pl.col("used_count") / pl.col("n_respondents"),
        wanted_pct=pl.col("wanted_count") / pl.col("n_respondents")
    )
    .select(["year", "tag", "used_pct", "wanted_pct", "n_respondents"])
    .sort(["year", "tag"])
)

## Sección 4 — Persistencia (Exportar a Parquet)

Finalmente, evaluamos el pipeline completo de operaciones perezosas materializándolo en memoria con `.collect(streaming=True)`. 
Posteriormente, el `DataFrame` resultante se exporta a un archivo de tipo Parquet, logrando así alta compresión y excelente rendimiento para la siguiente fase de análisis.

In [7]:
PROCESSED_DATA_DIR = Path("../data/datos_procesados/")
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

FINAL_PARQUET_PATH = PROCESSED_DATA_DIR / "survey_unificado.parquet"

# Ejecutamos de forma diferida todo el grafo de cálculo y activamos streaming para mayor eficiencia en memoria
df_final = final_lazy_query.collect(streaming=True)

# Inspeccionamos una muestra antes de guardar
print(f"Dimensiones del DataFrame final: {df_final.shape}")
print(df_final.head())

# Guardamos en Parquet con compresión zstd
df_final.write_parquet(FINAL_PARQUET_PATH, compression="zstd")

print(f"\n¡Proceso exitoso! Datos consolidados exportados en: {FINAL_PARQUET_PATH}")

C:\Users\PC MASTER\AppData\Local\Temp\ipykernel_12264\2980331184.py:7: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  df_final = final_lazy_query.collect(streaming=True)


Dimensiones del DataFrame final: (314, 5)
shape: (5, 5)
┌──────┬──────────┬──────────┬────────────┬───────────────┐
│ year ┆ tag      ┆ used_pct ┆ wanted_pct ┆ n_respondents │
│ ---  ┆ ---      ┆ ---      ┆ ---        ┆ ---           │
│ i32  ┆ str      ┆ f64      ┆ f64        ┆ u32           │
╞══════╪══════════╪══════════╪════════════╪═══════════════╡
│ 2017 ┆ assembly ┆ 0.035472 ┆ 0.037418   ┆ 51392         │
│ 2017 ┆ c        ┆ 0.135702 ┆ 0.094198   ┆ 51392         │
│ 2017 ┆ c#       ┆ 0.242762 ┆ 0.198222   ┆ 51392         │
│ 2017 ┆ c++      ┆ 0.158682 ┆ 0.148797   ┆ 51392         │
│ 2017 ┆ clojure  ┆ 0.007608 ┆ 0.025451   ┆ 51392         │
└──────┴──────────┴──────────┴────────────┴───────────────┘

¡Proceso exitoso! Datos consolidados exportados en: ..\data\datos_procesados\survey_unificado.parquet
